# Prétraitement Avancé et Modélisation

## Chargement des données

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("gestionlogistique").getOrCreate()
df = spark.read.option("header", "true").parquet("data/output_training_parquet/part-00000-97d86afc-357a-4d4f-96af-144a5770788b-c000.snappy.parquet")

df.show()
df.printSchema()

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/19 22:30:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/19 22:30:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------------+------------------+------------------+-------------------+------------------------+-------------------+------------------+----------------------+------------------+-----------------+-------------+------------------+----------------+-----------------+-------------------+-----------------+--------------------+-----------------------+
|Benefit per order|Sales per customer|Late_delivery_risk|Order Item Quantity|Order Item Product Price|Order Item Discount|  Order Item Total|Order Profit Per Order|          distance|Order Country_ohe|     Type_ohe|Customer State_ohe|Order Region_ohe|Shipping Mode_ohe|Department Name_ohe|Category Name_ohe|    numeric_features|scaled_numeric_features|
+-----------------+------------------+------------------+-------------------+------------------------+-------------------+------------------+----------------------+------------------+-----------------+-------------+------------------+----------------+-----------------+-------------------+-----

25/11/19 22:30:23 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Conversion des colonnes numériques

In [3]:
from pyspark.ml.feature import VectorAssembler

target_column = "Late_delivery_risk"

feature_cols = [
    "Benefit per order",
    "Sales per customer",
    "Order Item Quantity",
    "Order Item Product Price",
    "Order Item Discount",
    "Order Item Total",
    "Order Profit Per Order",
    "distance",
    "Order Country_ohe",
    "Type_ohe",
    "Customer State_ohe",
    "Order Region_ohe",
    "Shipping Mode_ohe",
    "Department Name_ohe",
    "Category Name_ohe",
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vec = assembler.transform(df).select("features", target_column)


## Random Forest AMÉLIORÉ avec hyperparamètres optimisés

In [4]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

train_v2, test_v2 = df_vec.randomSplit([0.8, 0.2], seed=42)

rf_improved = RandomForestClassifier(
    labelCol=target_column,
    featuresCol="features",
    numTrees=200,
    maxDepth=10,
    minInstancesPerNode=10,
    maxBins=64,
    subsamplingRate=0.8,
    seed=42
)

model_improved = rf_improved.fit(train_v2)
pred_improved = model_improved.transform(test_v2)

roc_improved = BinaryClassificationEvaluator(labelCol="Late_delivery_risk", metricName="areaUnderROC").evaluate(pred_improved)
accuracy_improved = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="accuracy").evaluate(pred_improved)
f1_improved = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="f1").evaluate(pred_improved)

print(f"AUC ROC = {roc_improved:.4f}")
print(f"Accuracy = {accuracy_improved:.4f}")
print(f"F1 Score = {f1_improved:.4f}")

25/11/19 22:30:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/11/19 22:31:44 WARN DAGScheduler: Broadcasting large task binary with size 1442.8 KiB
25/11/19 22:31:44 WARN DAGScheduler: Broadcasting large task binary with size 1442.8 KiB
25/11/19 22:31:59 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
25/11/19 22:31:59 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
25/11/19 22:32:16 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
25/11/19 22:32:16 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
25/11/19 22:32:35 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
25/11/19 22:32:35 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
25/11/19 22:32:56 WARN DAGScheduler: Broadcasting large task binary with size 5.2 MiB
25/11/19 22:32:56 WARN DAGSched

AUC ROC = 0.7500
Accuracy = 0.6934
F1 Score = 0.6913
